## Exercise 1

**Dataset Used:** Custom/Synthetic Array Data (numpy)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 9: Recurrent Neural Networks (RNN/LSTM)
# Niveau: Anfänger
# Aufgabe 1 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. Sinuswelle als Zeitreihe generieren ────────────────────────────────────
FENSTERGRÖSSE = 20    # Länge der Eingabesequenz
N_PUNKTE      = 2000  # Gesamtlänge der Zeitreihe

t = np.linspace(0, 4 * np.pi, N_PUNKTE)
zeitreihe = np.sin(t).astype("float32")

print(f"Zeitreihe: {N_PUNKTE} Datenpunkte, Fenstergröße: {FENSTERGRÖSSE}")

# ── 2. Überwachte Lernaufgabe: Fenster → nächster Wert ───────────────────────
def fenster_erstellen(reihe, fenstergr):
    """Sliding-Window: X = [t:t+win], y = [t+win]"""
    X, y = [], []
    for i in range(len(reihe) - fenstergr):
        X.append(reihe[i : i + fenstergr])
        y.append(reihe[i + fenstergr])
    return np.array(X), np.array(y)

X, y = fenster_erstellen(zeitreihe, FENSTERGRÖSSE)
print(f"X-Shape: {X.shape}, y-Shape: {y.shape}")

# RNN erwartet 3D-Eingabe: (n_samples, timesteps, features)
X = X[:, :, np.newaxis]

# Train-/Test-Split (80/20)
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# ── 3. SimpleRNN-Modell aufbauen ──────────────────────────────────────────────
modell = tf.keras.Sequential([
    tf.keras.layers.SimpleRNN(
        32, input_shape=(FENSTERGRÖSSE, 1),
        return_sequences=False, name="simple_rnn"
    ),
    tf.keras.layers.Dense(16, activation="relu", name="dense_1"),
    tf.keras.layers.Dense(1, name="ausgabe"),    # Regression: ein Wert
], name="SimpleRNN_Sinus")

modell.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)
modell.summary()

# ── 4. Training ───────────────────────────────────────────────────────────────
print("\nTrainiere SimpleRNN...")
history = modell.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

# ── 5. Vorhersagen ────────────────────────────────────────────────────────────
vorhersagen = modell.predict(X_test, verbose=0).flatten()

# Mittlerer absoluter Fehler
mae = np.mean(np.abs(vorhersagen - y_test))
print(f"\nTest-MAE: {mae:.6f}")

# ── 6. Visualisierung ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Vorhersagen vs. Tatsächliche Werte (erste 200 Testpunkte)
n_zeigen = 200
axes[0].plot(y_test[:n_zeigen],        label="Tatsächlich", linewidth=2)
axes[0].plot(vorhersagen[:n_zeigen],   label="SimpleRNN Vorhersage",
             linestyle="--", linewidth=2)
axes[0].set_title("SimpleRNN: Sinuswellen-Vorhersage")
axes[0].set_xlabel("Zeitschritt")
axes[0].set_ylabel("Amplitudenwert")
axes[0].legend()
axes[0].grid(True)

# Trainingsverlust
axes[1].plot(history.history["loss"],     label="Training-MSE")
axes[1].plot(history.history["val_loss"], label="Validierungs-MSE")
axes[1].set_title("Trainingsverlauf (Mean Squared Error)")
axes[1].set_xlabel("Epoche")
axes[1].set_ylabel("MSE")
axes[1].legend()
axes[1].grid(True)

plt.suptitle("SimpleRNN für Zeitreihenvorhersage (Sinuswelle)", fontsize=13)
plt.tight_layout()
plt.savefig("A9_1_simple_rnn_sinus.png", dpi=100)
plt.show()
print("Diagramm gespeichert: A9_1_simple_rnn_sinus.png")


## Exercise 2

**Dataset Used:** Custom/Synthetic Array Data (numpy)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 9: Recurrent Neural Networks (RNN/LSTM)
# Niveau: Anfänger
# Aufgabe 2 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. Sinuswelle generieren ──────────────────────────────────────────────────
FENSTERGRÖSSE = 20
N_PUNKTE      = 2000

t = np.linspace(0, 4 * np.pi, N_PUNKTE)
zeitreihe = np.sin(t).astype("float32")

def fenster_erstellen(reihe, fenstergr):
    X, y = [], []
    for i in range(len(reihe) - fenstergr):
        X.append(reihe[i : i + fenstergr])
        y.append(reihe[i + fenstergr])
    return np.array(X)[:, :, np.newaxis], np.array(y)

X, y = fenster_erstellen(zeitreihe, FENSTERGRÖSSE)
split   = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
print(f"Trainingsdaten: {X_train.shape}")

# ── 2. SimpleRNN-Modell ───────────────────────────────────────────────────────
def rnn_modell():
    return tf.keras.Sequential([
        tf.keras.layers.SimpleRNN(32, input_shape=(FENSTERGRÖSSE, 1), name="rnn"),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1),
    ], name="SimpleRNN")

# ── 3. LSTM-Modell ────────────────────────────────────────────────────────────
def lstm_modell():
    return tf.keras.Sequential([
        tf.keras.layers.LSTM(32, input_shape=(FENSTERGRÖSSE, 1), name="lstm"),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1),
    ], name="LSTM")

# ── 4. Modelle kompilieren und trainieren ─────────────────────────────────────
m_rnn  = rnn_modell()
m_lstm = lstm_modell()

for m in [m_rnn, m_lstm]:
    m.compile(optimizer="adam", loss="mse", metrics=["mae"])

print("\nTrainiere SimpleRNN...")
h_rnn = m_rnn.fit(X_train, y_train, epochs=20, batch_size=32,
                   validation_split=0.1, verbose=0)

print("Trainiere LSTM...")
h_lstm = m_lstm.fit(X_train, y_train, epochs=20, batch_size=32,
                     validation_split=0.1, verbose=0)

# ── 5. Vorhersagen ────────────────────────────────────────────────────────────
pred_rnn  = m_rnn.predict(X_test, verbose=0).flatten()
pred_lstm = m_lstm.predict(X_test, verbose=0).flatten()

mae_rnn  = np.mean(np.abs(pred_rnn  - y_test))
mae_lstm = np.mean(np.abs(pred_lstm - y_test))

print(f"\nSimpleRNN MAE: {mae_rnn:.6f}")
print(f"LSTM      MAE: {mae_lstm:.6f}")
print(f"LSTM-Verbesserung: {(mae_rnn - mae_lstm) / mae_rnn * 100:.1f}%")

# Parameteranzahl
print(f"\nSimpleRNN Parameter: {m_rnn.count_params():,}")
print(f"LSTM Parameter:      {m_lstm.count_params():,}")
print("(LSTM hat ~4× mehr Parameter durch Gate-Mechanismus)")

# ── 6. Visualisierung ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
n_zeigen = 200

# Vorhersagen
axes[0].plot(y_test[:n_zeigen],        label="Tatsächlich",  linewidth=2)
axes[0].plot(pred_rnn[:n_zeigen],      label=f"SimpleRNN (MAE={mae_rnn:.5f})",
             linestyle="--", linewidth=1.5)
axes[0].plot(pred_lstm[:n_zeigen],     label=f"LSTM (MAE={mae_lstm:.5f})",
             linestyle="-.", linewidth=1.5)
axes[0].set_title("SimpleRNN vs. LSTM: Sinuswellen-Vorhersage")
axes[0].set_xlabel("Zeitschritt")
axes[0].set_ylabel("Amplitudenwert")
axes[0].legend()
axes[0].grid(True)

# Trainingsverlauf
axes[1].plot(h_rnn.history["val_loss"],  label="SimpleRNN (Val-MSE)", linewidth=2)
axes[1].plot(h_lstm.history["val_loss"], label="LSTM (Val-MSE)",      linewidth=2)
axes[1].set_title("Validierungs-Verlust (MSE)")
axes[1].set_xlabel("Epoche")
axes[1].set_ylabel("MSE")
axes[1].legend()
axes[1].grid(True)

plt.suptitle("SimpleRNN vs. LSTM: Langzeit-Gedächtnis bei Sinuswellen", fontsize=13)
plt.tight_layout()
plt.savefig("A9_2_lstm_vs_rnn.png", dpi=100)
plt.show()
print("Diagramm gespeichert: A9_2_lstm_vs_rnn.png")


## Exercise 3

**Dataset Used:** Custom/Synthetic Array Data (numpy)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 9: Recurrent Neural Networks (RNN/LSTM)
# Niveau: Anfänger
# Aufgabe 3 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import random

print("TensorFlow Version:", tf.__version__)

# ── 1. Synthetischen Sentiment-Datensatz erstellen ───────────────────────────
POSITIVE_WOERTER = [
    "gut", "toll", "super", "ausgezeichnet", "wunderbar", "fantastisch",
    "klasse", "prima", "hervorragend", "brilliant", "freude", "liebe",
    "empfehle", "begeistert", "perfekt", "angenehm", "glücklich"
]
NEGATIVE_WOERTER = [
    "schlecht", "schrecklich", "furchtbar", "enttäuschend", "miserabel",
    "langweilig", "nervig", "awful", "terrible", "problematisch",
    "enttäuscht", "frustrierend", "hässlich", "falsch", "schade"
]
NEUTRALE_FUELLER = [
    "das", "ein", "ist", "war", "hat", "ich", "sehr", "wirklich",
    "und", "oder", "der", "die", "es", "habe", "noch", "auch"
]

def rezension_generieren(sentiment, laenge=10):
    """Synthetische kurze Rezension (0=negativ, 1=positiv)."""
    if sentiment == 1:
        schluessel = random.choices(POSITIVE_WOERTER, k=laenge // 3)
    else:
        schluessel = random.choices(NEGATIVE_WOERTER, k=laenge // 3)
    fueller = random.choices(NEUTRALE_FUELLER, k=laenge - len(schluessel))
    woerter = schluessel + fueller
    random.shuffle(woerter)
    return woerter

N_PROBEN = 2000
random.seed(42)
np.random.seed(42)

texte    = []
labels   = []
for _ in range(N_PROBEN // 2):
    texte.append(rezension_generieren(1, laenge=12))
    labels.append(1)
    texte.append(rezension_generieren(0, laenge=12))
    labels.append(0)

print(f"Beispiel positiv: {texte[0]}")
print(f"Beispiel negativ: {texte[1]}")

# ── 2. Tokenisierung ─────────────────────────────────────────────────────────
alle_woerter = sorted(set(w for text in texte for w in text))
wort_zu_idx  = {w: i+1 for i, w in enumerate(alle_woerter)}  # 0 = Padding
VOKABULAR_GR = len(wort_zu_idx) + 1

SEQUENZ_LAENGE = 12
sequenzen = []
for text in texte:
    seq = [wort_zu_idx.get(w, 0) for w in text[:SEQUENZ_LAENGE]]
    # Padding auf feste Länge
    while len(seq) < SEQUENZ_LAENGE:
        seq.append(0)
    sequenzen.append(seq)

X = np.array(sequenzen, dtype="int32")
y = np.array(labels,    dtype="float32")

# Mischen
idx = np.random.permutation(len(X))
X, y = X[idx], y[idx]

# Train-/Test-Split
split   = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
print(f"\nVokabular-Größe: {VOKABULAR_GR}")
print(f"Trainingsdaten: {X_train.shape}  Labels: {y_train.shape}")

# ── 3. LSTM-Klassifikator aufbauen ────────────────────────────────────────────
modell = tf.keras.Sequential([
    # Embedding: wandelt Token-IDs in dichte Vektoren um
    tf.keras.layers.Embedding(
        input_dim=VOKABULAR_GR,
        output_dim=32,
        input_length=SEQUENZ_LAENGE,
        mask_zero=True,     # ignoriert Padding-Tokens
        name="embedding"
    ),
    # LSTM: verarbeitet die Sequenz und gibt einen Zustandsvektor zurück
    tf.keras.layers.LSTM(32, name="lstm"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid", name="ausgabe"),
], name="Sentiment_LSTM")

modell.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
modell.summary()

# ── 4. Training ───────────────────────────────────────────────────────────────
print("\nTrainiere LSTM-Sentiment-Klassifikator...")
history = modell.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

# ── 5. Evaluation ─────────────────────────────────────────────────────────────
test_loss, test_acc = modell.evaluate(X_test, y_test, verbose=0)
print(f"\nTest-Verlust:     {test_loss:.4f}")
print(f"Test-Genauigkeit: {test_acc:.4f}")

# ── 6. Manuelle Vorhersagen ───────────────────────────────────────────────────
print("\nBeispiel-Vorhersagen:")
for i in range(5):
    wirklicher_text = " ".join([w for w in texte[split+i]])
    pred = modell.predict(X_test[i:i+1], verbose=0)[0, 0]
    sentiment_str = "POSITIV" if pred > 0.5 else "NEGATIV"
    wahr_str      = "POSITIV" if y_test[i] == 1 else "NEGATIV"
    korrekt       = "✓" if (pred > 0.5) == (y_test[i] == 1) else "✗"
    print(f"  Text: '{wirklicher_text[:50]}...'")
    print(f"  → Pred: {sentiment_str} ({pred:.3f}), Wahr: {wahr_str} {korrekt}")

# ── 7. Trainingsverlauf plotten ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["loss"],     label="Training")
axes[0].plot(history.history["val_loss"], label="Validierung")
axes[0].set_title("Verlust (Binary Cross-Entropy)")
axes[0].set_xlabel("Epoche")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history["accuracy"],     label="Training")
axes[1].plot(history.history["val_accuracy"], label="Validierung")
axes[1].set_title("Genauigkeit")
axes[1].set_xlabel("Epoche")
axes[1].legend()
axes[1].grid(True)

plt.suptitle("LSTM Textkategorisierung (Synthetisches Sentiment-Dataset)", fontsize=13)
plt.tight_layout()
plt.savefig("A9_3_lstm_sentiment.png", dpi=100)
plt.show()
print("Diagramm gespeichert: A9_3_lstm_sentiment.png")
